# Load packages

In [50]:
import os
import ee
import geemap
import json
import requests
import folium

from IPython.display import display, HTML, clear_output

# Initialise GEE

You will need to authenticate your Google account the first time you run this. 

In [51]:
# Initialize Earth Engine
try:
    ee.Initialize()
    print("Earth Engine already initialized")
except Exception as e:
    ee.Authenticate()
    ee.Initialize()
    print("Earth Engine initialized")

Earth Engine already initialized


## Plot the basemap

The first time you run the code below, follow the link to authenticate your Google account. After following the instructions online, you will receive an authorization code, which you can paste back into the input box in your notebook. If you're using VSCode, this will appear at the top of your window where the search bar is.

In [52]:
Map = geemap.Map(lite_mode=True, zoom=2)
Map.add_basemap("SATELLITE")
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

## Generate AOI

There are several approaches to create areas of interest (AOIs).

### Manually create an AOI from coordinates

In [53]:
# Alternative: Define AOI from coordinates
# aoi = ee.Geometry.Rectangle([-122.5, 37.5, -122.0, 38.0])  # San Francisco area

### Function to read an AOI from file

In [54]:
# Option 1: Read JSON from file
def load_aoi_from_file(json_file_path):
    with open(json_file_path, 'r') as f:
        geojson = json.load(f)
    
    # Create an ee.Geometry from the GeoJSON
    return ee.Geometry(geojson)

### Specify a pre-created AOI file

In [55]:
# Load AOI from file
aoi_name = 'mimal_test'
aoi = load_aoi_from_file(fr'AOIs/{aoi_name}.geojson')

# New basemap for AOI
Map2 = geemap.Map(lite_mode=True, zoom=2)
Map2.add_basemap("SATELLITE")
Map2

# Show the AOI on map
Map2.addLayer(aoi, {}, 'Area of Interest')
# Center the map on the AOI and zoom to it
Map2.centerObject(aoi, zoom=10)  # Adjust zoom level (1-20) as needed
# Map  # Display the map with the AOI

Map2

Map(center=[-13.418568982735165, 134.6245295479214], controls=(WidgetControl(options=['position', 'transparent…

### Function for multiple features in the json 

This function allows for filtering to just select certain features. Here we just want the 'North Australian Tropical Savanna'.

In [56]:
def load_aoi_from_file(json_file_path, filter_field=None, filter_value=None):
    with open(json_file_path) as f:
        geojson = json.load(f)
    
    gtype = geojson.get('type')
    if gtype == 'FeatureCollection':
        features = [ee.Feature(feat) for feat in geojson['features']]
        fc = ee.FeatureCollection(features)
        if filter_field is not None:
            fc = fc.filter(ee.Filter.eq(filter_field, filter_value))
        return fc.geometry()
    elif gtype == 'Feature':
        return ee.Geometry(geojson['geometry'])
    else:
        return ee.Geometry(geojson)

### Load and plot the Tropical Savanna AOI

In [57]:
# Load AOI from file
aoi_name = 'north_aus_tropical_savanna_buffer10km_wgs84'

aoi = load_aoi_from_file(
    f'AOIs/{aoi_name}.geojson'
)

Map2 = geemap.Map(lite_mode=True)
Map2.add_basemap("SATELLITE")
# Map2.addLayer(aoi, {}, 'Area of Interest')
# add the json directly (not the one on the GEE server-side)
Map2.add_geojson(f'AOIs/{aoi_name}.geojson', layer_name='Area of Interest')
Map2.set_center(133.0, -15.0, 5)  # lon, lat, zoom — roughly central North Australia
Map2

Map(center=[-15.0, 133.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

## Randomly sample AOIs

Because we want the training data to cover a large area that we will predict over, we can randomly sample images from across the prediction extent.

In [58]:
half_side_m = 5000  # half the side length in metres

# 2. Sample N random points within the shrunken region
n_points = 100
random_points = ee.FeatureCollection.randomPoints(
    region=aoi,
    points=n_points,
    seed=42,
)

# 3. Turn each point into a rectangular AOI
def point_to_rect(feature):
    return feature.setGeometry(
        feature.geometry().buffer(half_side_m).bounds()
    )

rect_fc = random_points.map(point_to_rect)

### Plot the sampled AOIs

In [ ]:
# Style: red outlines, no fill
Map2.addLayer(
    rect_fc.style(color='red', fillColor='FF000033', width=2),  # last 2 hex digits = alpha
    {},
    'Sampled AOIs',
)

Map2

Map(bottom=812.0, center=[-15.0, 133.0], controls=(WidgetControl(options=['position', 'transparent_bg'], posit…

# Get Sentinel-2 Collection with Cloud Masking

This function creates a cloud mask for Sentinel-2 imagery.

There are other approaches to filter clouds, such as the approach listed here: [https://developers.google.com/earth-engine/tutorials/community/sentinel-2-s2cloudless](https://developers.google.com/earth-engine/tutorials/community/sentinel-2-s2cloudless).

In [60]:
def maskS2clouds_CSPlus(image):
    """
    Mask Sentinel-2 using Cloud Score+.
    Assumes the CS+ bands ('cs', 'cs_cdf') have already been linked
    to the S2 collection via linkCollection().
    """
    # Use the cs_cdf band (cumulative distribution function variant)
    # is generally more robust than 'cs' for time-series work.
    # Threshold range: 0 (not clear) to 1 (clear).
    #   0.60 = permissive (more pixels kept, some haze/thin cloud)
    #   0.65 = balanced (Google's commonly recommended default)
    #   0.80+ = strict (clean composites, fewer observations)
    QA_BAND = 'cs_cdf'
    CLEAR_THRESHOLD = 0.65

    mask = image.select(QA_BAND).gte(CLEAR_THRESHOLD)
    return (image.divide(10000)
                 .updateMask(mask)
                 .copyProperties(image, ['system:time_start']))

# Get Monthly Composites of Sentinel-2

## Define a function to get monthly composites

We want to create monthly composites of Sentinel-2 imagery. This function will filter the Sentinel-2 image collection by date and area of interest (AOI), apply the cloud mask, and then compute the median of each 'cloud-free' pixel for the month.

In [61]:
def get_monthly_composites(start_date, end_date, aoi):
    """Generate monthly S2 composites using Cloud Score+ for cloud/shadow masking."""
    start = ee.Date(start_date)
    end = ee.Date(end_date)
    months = end.difference(start, 'month').round().int()

    # Load harmonized S2 SR and link Cloud Score+ bands once.
    # Pre-filter by AOI, date, and a generous cloud cover cap to drop the worst scenes
    # before the join.
    s2_base = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                 .filterBounds(aoi)
                 .filterDate(start_date, end_date)
                 .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 90)))

    csPlus = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
    s2_linked = s2_base.linkCollection(csPlus, ['cs', 'cs_cdf'])

    raw_count = s2_base.size().getInfo()
    print(f"  Found {raw_count} S2 images for {start_date} to {end_date}")

    def get_monthly_image(month_index):
        current_month_start = start.advance(month_index, 'month')
        current_month_end = current_month_start.advance(1, 'month')
        date_format = current_month_start.format('YYYY-MM')

        # Filter the already-linked collection to this month and apply CS+ mask
        monthly = (s2_linked
                     .filterDate(current_month_start, current_month_end)
                     .map(maskS2clouds_CSPlus))

        composite = monthly.median().clip(aoi)

        # Bands are already scaled to 0–1 by maskS2clouds_CSPlus
        rgb = composite.select(['B2', 'B3', 'B4'])

        return rgb.set({
            'system:index': date_format,
            'system:time_start': current_month_start.millis()
        })

    month_indices = ee.List.sequence(0, months.subtract(1))
    composites = ee.ImageCollection.fromImages(month_indices.map(get_monthly_image))
    return composites

## Define a function to export and download images

If the images are small enough, you can download them directly to your local machine. If they are too large, you can export them to your Google Drive.

In [62]:
# For larger or higher resolution images, use Google Drive export:
def export_to_drive(composites, aoi, aoi_name, drive_folder='Earth_Engine_Exports', scale=10):
    image_list = composites.toList(composites.size())
    num_images = image_list.size().getInfo()
    tasks = []

    for i in range(num_images):
        image = ee.Image(image_list.get(i))
        date_str = image.get('system:index').getInfo()

        task = ee.batch.Export.image.toDrive(
            image=image,                           # raw reflectance, not visualized
            description=f'{aoi_name}_{date_str}',
            folder=drive_folder,
            fileNamePrefix=f'{aoi_name}_{date_str}',
            scale=scale,
            region=aoi,
            crs='EPSG:4326',
            maxPixels=1e10,
        )
        task.start()
        tasks.append(task)
        print(f"Started: {aoi_name}_{date_str}")

    return tasks  # return so you can monitor with task.status()

# Download images for a specified date range

We will get an image for June 2024 as an example.

In [63]:
# Define parameters
start_date = '2024-06-01'
end_date = '2024-06-30' 
output_folder = 'image/sentinel2_monthly_images'

### Run the function

The function will print how many suitable images were found for the specified date range and AOI. If no images are found, you may need to adjust your date range or AOI.

In [64]:
# Grab the first randomly sampled AOI's geometry
single_aoi = ee.Feature(rect_fc.first()).geometry()

# Get monthly composites
composites = get_monthly_composites(start_date, end_date, single_aoi)

  Found 12 S2 images for 2024-06-01 to 2024-06-30


## Plot one of the randomly sampled AOIs

In [65]:
# Visualize a preview
map2 = geemap.Map()
map2.centerObject(single_aoi, zoom=12)
map2.add_basemap("SATELLITE")
map2.addLayer(single_aoi, {}, 'Area of Interest')

# Add the first image to the map
first_image = ee.Image(composites.first())
viz_params = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.25}
# map2.addLayer(first_image, viz_params, 'First Monthly Composite')
# map2

# Get the number of images in the collection
count = composites.size().getInfo()
print(f"Found {count} monthly composites")

# Add each monthly composite to the map with its date as the layer name
if count > 0:
    # Convert to list for iterating
    date_strs = composites.aggregate_array('system:index').getInfo()
    image_list = composites.toList(count)
    
    # Add each image as a separate layer
    for i, date_str in enumerate(date_strs):
        image = ee.Image(image_list.get(i))
        map2.addLayer(image, viz_params, f'S2 RGB {date_str}', shown=(i == 0))
           
    print(f"Added {count} layers to the map. Use the layer control to toggle visibility.")
else:
    print("No composites available to display.")

# Add the layer control (if not already visible)
map2.add_layer_control()

# Display the map
map2

Found 1 monthly composites
Added 1 layers to the map. Use the layer control to toggle visibility.


Map(center=[-15.993129456014232, 142.00998969735878], controls=(WidgetControl(options=['position', 'transparen…

In [66]:
# Convert the FeatureCollection to a list and get the count
rect_list = rect_fc.toList(rect_fc.size())
n = rect_fc.size().getInfo()

print(f'Processing {n} AOIs...')

all_tasks = []
base_name = aoi_name  # preserve the original

for i in range(n):
    single_aoi = ee.Feature(rect_list.get(i)).geometry()
    single_name = f'{base_name}_{i:03d}'   
    
    print(f'\n[{i+1}/{n}] AOI: {single_name}')

    try:
        composites = get_monthly_composites(start_date, end_date, single_aoi)
        tasks = export_to_drive(
            composites,
            aoi=single_aoi,
            aoi_name=single_name,
            drive_folder='sentinel2_images',
            scale=10,
        )
        all_tasks.extend(tasks)
    except Exception as e:
        print(f'  FAILED for {single_name}: {e}')
        continue

print(f'\nStarted {len(all_tasks)} total export tasks.')

Processing 100 AOIs...

[1/100] AOI: north_aus_tropical_savanna_buffer10km_wgs84_000
  Found 12 S2 images for 2024-06-01 to 2024-06-30
Started: north_aus_tropical_savanna_buffer10km_wgs84_000_2024-06

[2/100] AOI: north_aus_tropical_savanna_buffer10km_wgs84_001
  Found 24 S2 images for 2024-06-01 to 2024-06-30
Started: north_aus_tropical_savanna_buffer10km_wgs84_001_2024-06

[3/100] AOI: north_aus_tropical_savanna_buffer10km_wgs84_002
  Found 12 S2 images for 2024-06-01 to 2024-06-30
Started: north_aus_tropical_savanna_buffer10km_wgs84_002_2024-06

[4/100] AOI: north_aus_tropical_savanna_buffer10km_wgs84_003
  Found 6 S2 images for 2024-06-01 to 2024-06-30
Started: north_aus_tropical_savanna_buffer10km_wgs84_003_2024-06

[5/100] AOI: north_aus_tropical_savanna_buffer10km_wgs84_004
  Found 20 S2 images for 2024-06-01 to 2024-06-30
Started: north_aus_tropical_savanna_buffer10km_wgs84_004_2024-06

[6/100] AOI: north_aus_tropical_savanna_buffer10km_wgs84_005
  Found 6 S2 images for 2024-06

### To check the status of the downloads (from the server-side)

In [67]:
from collections import Counter
states = Counter(t.status()['state'] for t in all_tasks)
print(states)   

Counter({'READY': 66, 'COMPLETED': 32, 'RUNNING': 2})


### If you need to kill all tasks

In [ ]:
# for t in all_tasks:
#     if t.status()['state'] in ('READY', 'RUNNING'):
#         t.cancel()